# Model Building — Credit Risk Analytics

**Author:** Machine Learning Engineer, Horizon Lending Inc.  
**Date:** July 2026  
**Target:** `TARGET` (1 = default, 0 = repaid, ~8% default rate)  
**Features:** ~150–250 engineered features (from `src.preprocess`)  

---

## Objective

Train, tune, compare, and select the best model for predicting loan default. This notebook orchestrates the training and evaluation pipeline by calling `src.preprocess`, `src.train`, and `src.evaluate`. No model code lives in this notebook — it delegates to reusable modules.

## Workflow

```
1. Load preprocessed data (from src.preprocess.build_feature_pipeline)
2. Split: 64% train / 16% validation / 20% test (stratified)
3. Train 5 candidate models with default params
4. Evaluate on validation set → build leaderboard
5. Tune top-2 models (Optuna, 50 trials each)
6. Select best model → retrain on train+val
7. Final evaluation on held-out test set
8. Threshold tuning & calibration
9. Generate plots, model card, and save artifacts
```

---

## Imports & Configuration

All imports come from the `src` package — no raw sklearn calls in the notebook.

**Notebook imports:**

```python
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Project modules (all training/evaluation logic lives here)
from src.preprocess import build_feature_pipeline
from src.train import (
    ModelTrainer, DataSplitter, CrossValidator, ModelLeaderboard,
    ModelPersister, DEFAULT_TRAIN_CONFIG
)
from src.evaluate import (
    ModelEvaluator, ThresholdTuner, CalibrationAssessor,
    EvaluationVisualizer, ModelComparer, ModelCardGenerator,
    DEFAULT_EVAL_CONFIG
)

# Optional: experiment tracking
# import mlflow  (for production tracking)
```

**Key constraint:** The test set (`X_test`, `y_test`) is NEVER touched until the final evaluation cell. All model selection, threshold tuning, and calibration use only `X_train`/`y_train`/`X_val`/`y_val`.

---

## Section 1: Load & Split Data

Load the master feature matrix produced by `src.preprocess.build_feature_pipeline()` and split into train / validation / test.

```python
# Load preprocessed data
pipeline = build_feature_pipeline()
X = pipeline.fit_transform(train_tables)  # or load from parquet
y = X.pop('TARGET')

# Stratified split
splitter = DataSplitter(test_size=0.20, val_size=0.20, random_state=42)
splits = splitter.split(X, y)

X_train, X_val, X_test = splits['X_train'], splits['X_val'], splits['X_test']
y_train, y_val, y_test = splits['y_train'], splits['y_val'], splits['y_test']

print(f'Train:   {X_train.shape}, defaults: {y_train.sum():,} ({y_train.mean():.1%})')
print(f'Val:     {X_val.shape},   defaults: {y_val.sum():,} ({y_val.mean():.1%})')
print(f'Test:    {X_test.shape},  defaults: {y_test.sum():,} ({y_test.mean():.1%})')
```

**Why this split?**
- 64/16/20 is standard for medium-sized datasets (~300K rows).
- Stratified split preserves the ~8% default rate in all three sets.
- The validation set is large enough (~48K rows) for reliable threshold tuning.
- The test set (~60K rows) provides tight confidence intervals on final metrics.

---

## Section 2: Model Candidates

Five models are trained and compared. Each is documented below with its rationale, trade-offs, and expected performance on this dataset.

### 2.1 Logistic Regression

**Why choose it:** Logistic regression is the industry-standard baseline for credit scoring. It is fully interpretable (coefficients = log-odds contribution per feature), fast to train, and produces well-calibrated probabilities natively. Many traditional bank scorecards (FICO-based) are built on logistic regression.

**Advantages:**
- Every feature gets a signed coefficient — you can explain exactly why a borrower's PD is high or low.
- Naturally calibrated (minimizes log loss, not accuracy).
- Converts directly to a regulatory-grade scorecard via scaling.
- Very low variance; hard to overfit with L2 regularization.
- Training time < 30 seconds on 300K × 200 features.

**Disadvantages:**
- Assumes a linear relationship between log-odds and features.
- Cannot capture interactions unless explicitly engineered (but we already added interaction features in Group J).
- Requires feature scaling (handled by `FeatureScaler` in the pipeline).
- Lower AUC ceiling compared to tree-based models (typically 0.72–0.76).

**Configuration:**
- L2 penalty (ridge), `C=0.1`, `solver='saga'`, `class_weight='balanced'`.
- Wrapped in a pipeline with `RobustScaler`.

**Expected AUC:** 0.72 – 0.76 (baseline)

### 2.2 Decision Tree

**Why choose it:** A single decision tree is the simplest tree-based model. It captures non-linear relationships and interactions natively, requires no feature scaling, and is fully interpretable (you can visualize the entire tree). It serves as a baseline to demonstrate why ensemble methods are superior.

**Advantages:**
- Intuitive for stakeholder communication: "If income < $40K and external score < 0.4, deny."
- Handles non-linearity, missing values, and feature interactions.
- No feature scaling or encoding needed.

**Disadvantages:**
- Very high variance — a small change in training data can produce a completely different tree.
- Poor generalization compared to ensembles (bagging, boosting).
- Probability estimates are unreliable (based on small leaf-node proportions).
- Tends to overfit even with aggressive pruning.

**Configuration:**
- `max_depth=8`, `min_samples_leaf=50`, `class_weight='balanced'`.
- Pruned enough to avoid overfitting on 300K rows.

**Expected AUC:** 0.62 – 0.68 (weak, by design — it is the "worst acceptable" baseline)

### 2.3 Random Forest

**Why choose it:** Random Forest aggregates hundreds of decorrelated decision trees via bagging and random feature selection. It dramatically reduces variance compared to a single tree while maintaining the ability to model non-linear relationships and interactions.

**Advantages:**
- Excellent out-of-the-box performance with minimal tuning.
- Robust to outliers, irrelevant features, and missing data.
- Provides reliable feature importance rankings (mean decrease in impurity).
- Parallelizable — trains efficiently on multi-core machines.
- Less prone to overfitting than boosted models.

**Disadvantages:**
- Not as interpretable as a single tree or logistic regression.
- Large model size (200+ trees) — memory and disk footprint.
- Cannot extrapolate outside training range (vs. linear models).
- Typically outperformed by gradient boosting (XGBoost, LightGBM) on tabular data.
- Slower inference (200 trees vs. 1 boosted model with early stopping).

**Configuration:**
- `n_estimators=200`, `max_depth=12`, `min_samples_leaf=20`.
- `class_weight='balanced_subsample'` (adjusts weights per bootstrap sample).

**Expected AUC:** 0.76 – 0.80 (strong baseline)

### 2.4 XGBoost

**Why choose it:** XGBoost is the industry gold standard for tabular/structured data. It builds trees sequentially, each correcting the errors of the previous one, with built-in L1/L2 regularization and native missing-value handling. It has dominated Kaggle competitions and is widely deployed in production credit risk systems.

**Advantages:**
- Consistently achieves the highest AUC among single models on structured tabular data.
- Native handling of missing values (learns the optimal split direction for NaN).
- Built-in regularization (`reg_alpha`, `reg_lambda`) prevents overfitting.
- Early stopping with a separate validation set automatically determines optimal number of trees.
- Feature importance via gain, cover, and frequency.
- Optimized C++ backend; GPU support available.

**Disadvantages:**
- Many hyperparameters to tune (max_depth, learning_rate, subsample, colsample_bytree, etc.).
- Training is slower than LightGBM on datasets with many features.
- `scale_pos_weight` must be carefully set for imbalanced data.
- Can overfit if tuned too aggressively or if early stopping is disabled.
- Calibration may be poor despite high AUC (requires post-hoc calibration).

**Configuration (default):**
- `n_estimators=500`, `max_depth=6`, `learning_rate=0.05`
- `subsample=0.8`, `colsample_bytree=0.8`
- `scale_pos_weight = (n_negative / n_positive)` ≈ 11.5
- `early_stopping_rounds=50` on validation set

**Tuning strategy (Optuna):**
- Search over: `max_depth` (3–12), `learning_rate` (1e-3–0.3), `subsample` (0.5–1.0),
  `colsample_bytree` (0.3–1.0), `min_child_weight` (1–20), `reg_alpha` (1e-4–10), `reg_lambda` (1e-4–10).
- 50 trials with 3-fold CV AUC as objective. Median pruner drops unpromising trials early.

**Expected AUC:** 0.79 – 0.83 (top contender)

### 2.5 LightGBM

**Why choose it:** LightGBM is a gradient boosting framework that grows trees leaf-wise (instead of level-wise like XGBoost), which allows it to converge faster and often achieve better accuracy. It uses histogram-based binning for faster training and lower memory usage on wide datasets.

**Advantages:**
- Significantly faster training than XGBoost on datasets with 200+ features and 300K rows.
- Lower memory usage due to histogram-based binning.
- Native categorical feature support (can handle high-cardinality categories directly).
- Often achieves comparable or slightly better AUC than XGBoost on tabular data.
- Leaf-wise growth is more efficient at learning complex patterns.

**Disadvantages:**
- Leaf-wise growth can overfit on small datasets `num_leaves` must be constrained.
- Native categorical support requires preprocessing (`convert_dtypes`).
- More sensitive to hyperparameter choices than XGBoost.
- Less mature ecosystem (though the gap has narrowed significantly).
- `verbose=-1` is recommended to avoid excessive logging.

**Configuration (default):**
- `n_estimators=500`, `num_leaves=31`, `learning_rate=0.05`
- `subsample=0.8`, `colsample_bytree=0.8`
- `class_weight='balanced'`, `metric='auc'`
- `early_stopping_rounds=50`

**Tuning strategy (Optuna):**
- Search over: `num_leaves` (15–127), `learning_rate` (1e-3–0.3), `subsample` (0.5–1.0),
  `colsample_bytree` (0.3–1.0), `min_child_samples` (5–100), `reg_alpha` (1e-4–10), `reg_lambda` (1e-4–10).
- 50 trials with 3-fold CV AUC as objective.

**Expected AUC:** 0.79 – 0.83 (comparable to XGBoost, potentially faster)

---

## Section 3: Model Comparison

### 3.1 Default-Parameter Training

```python
from src.train import get_default_params, LogisticRegressionModel, \
    DecisionTreeModel, RandomForestModel, XGBoostModel, LightGBMModel

models = {
    'LogisticRegression': LogisticRegressionModel(params).build(),
    'DecisionTree':       DecisionTreeModel(params).build(),
    'RandomForest':       RandomForestModel(params).build(),
    'XGBoost':            XGBoostModel(params).build(),
    'LightGBM':           LightGBMModel(params).build(),
}

leaderboard = ModelLeaderboard().build_leaderboard(models, X_val, y_val)
leaderboard.sort_values('roc_auc', ascending=False)
```

**Expected leaderboard (default params):**

| Model | AUC | Avg Precision | Log Loss | Time (s) |
|---|---|---|---|---|
| XGBoost | 0.808 | 0.41 | 0.265 | 180 |
| LightGBM | 0.805 | 0.40 | 0.267 | 90 |
| RandomForest | 0.785 | 0.37 | 0.272 | 240 |
| LogisticRegression | 0.740 | 0.30 | 0.281 | 20 |
| DecisionTree | 0.650 | 0.20 | 0.295 | 15 |

*(Note: these are illustrative. Actual values depend on the feature set and data quality.)*

### 3.2 Hyperparameter Tuning (Top-2 Models)

Based on the leaderboard, the top-2 models (likely XGBoost and LightGBM) proceed to hyperparameter tuning via Optuna.

```python
from src.train import HyperparameterTuner

tuner = HyperparameterTuner(n_trials=50, cv_folds=3)

# Tune XGBoost
xgb_best_params, xgb_study = tuner.tune_xgboost(X_train, y_train, X_val, y_val)

# Tune LightGBM
lgb_best_params, lgb_study = tuner.tune_lightgbm(X_train, y_train, X_val, y_val)
```

**Optuna study visualization:**
- Parameter importance plot: which hyperparameters most affect AUC?
- Optimization history: does AUC plateau by trial 50?
- Parallel coordinate plot: which parameter combinations yield high AUC?

**Expected lift from tuning:** +0.005 to +0.015 AUC (modest but meaningful).

### 3.3 Model Selection & Retraining

The best model (highest validation AUC, considering training time as a tiebreaker) is selected, retrained on `X_train + X_val`, and evaluated on `X_test`.

```python
from src.train import ModelTrainer

trainer = ModelTrainer(config)
results = trainer.run(X_train, y_train)

best_model = results['best_model']
best_model_name = results['best_model_name']
leaderboard = results['leaderboard']
experiment_path = results['experiment_path']

print(f'Selected model: {best_model_name}')
print(f'Experiment saved to: {experiment_path}')
```

---

## Section 4: Evaluation Strategy

### 4.1 Test-Set Evaluation (One-Time, Final)

```python
from src.evaluate import ModelEvaluator

evaluator = ModelEvaluator()

# Get predictions
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Full evaluation
eval_results = evaluator.evaluate(y_test, y_pred_proba)

# Tune threshold on VALIDATION set, apply on TEST set
optimal_threshold = evaluator.tune_threshold(y_val, y_val_pred_proba, method='fbeta', beta=2.0)
print(f'Optimal threshold (F2-optimal): {optimal_threshold:.3f}')
```

### 4.2 Metrics Breakdown

| Metric | Formula / Definition | Why It Matters | Expected |
|---|---|---|---|
| **ROC-AUC** | Area under TPR vs. FPR curve | Overall rank-ordering ability of the model | ≥ 0.80 |
| **Gini** | `2 × AUC − 1` | Banking-standard metric; proportional to AUC | ≥ 0.60 |
| **KS Statistic** | `max(TPR − FPR)` over all thresholds | Maximum separation power; regulatory reporting | ≥ 0.45 |
| **Average Precision** | Area under precision-recall curve | More informative than AUC for imbalanced data | ≥ 0.35 |
| **Precision@5%** | Precision among highest-risk 5% of applicants | Collections: "Of our top-50 flagged borrowers, how many actually default?" | ≥ 0.40 |
| **Recall@5%** | Recall within highest-risk 5% | "What fraction of all defaults do we catch in the top 5%?" | ≥ 30% |
| **Brier Score** | Mean squared error: `(y_true − y_pred)^2` | Calibration quality; lower is better | < 0.08 |
| **Log Loss** | `−[y·log(p) + (1−y)·log(1−p)]` | Strict probability scoring rule | < 0.28 |
| **F2 Score** | Beta=2: weights recall 2× precision | Business: catching defaults matters more than false alarms | Best at tuned threshold |

### 4.3 Cross-Validation (Internal Consistency)

In addition to the single train/val/test split, the final model is evaluated with 5-fold stratified cross-validation on the full training set to verify stability.

```python
from src.train import CrossValidator

cv = CrossValidator(n_folds=5, scoring=['roc_auc', 'average_precision'])
cv_scores = cv.cross_validate(best_model, X_train, y_train)

print(f'CV AUC: {cv_scores["roc_auc"]["mean"]:.4f} ± {cv_scores["roc_auc"]["std"]:.4f}')
```

**Acceptance criteria:**
- CV AUC std < 0.01 (model is stable across folds)
- Test AUC within 0.02 of CV mean AUC (no overfitting/underfitting)
- CV and test AUC both ≥ 0.80

### 4.4 Threshold Tuning (Detailed)

The default `0.5` threshold is never optimal for imbalanced credit data. We search thresholds from 0.01 to 0.50 and optimize for business utility.

**Method 1: F2 Score (Primary)**
- Beta = 2: recall is twice as important as precision.
- Rationale: missing a default (FN) is more costly than a false alarm (FP).
- A false alarm costs a phone call (~$50). A missed default costs the loan principal (~$10K).

**Method 2: Youden's J**
- `J = sensitivity + specificity − 1`
- Maximizes balanced accuracy; useful as a reference without cost assumptions.

**Method 3: Profit-Based (Business-Aligned)**
- For each threshold, compute total portfolio profit:
  - Approve if `PD_i < threshold`: expected profit = `(1−PD_i) × revenue − PD_i × LGD`  
  - Decline if `PD_i >= threshold`: profit = 0
- Assumptions: 15% interest revenue, 60% LGD

```python
# Threshold tuning
tuner = ThresholdTuner(method='profit', business_assumptions={
    'revenue_per_loan_pct': 0.15,
    'loss_per_default_pct': 0.60,
})
optimal_threshold = tuner.tune(y_val, y_val_proba)

# Visualize trade-offs
tuner.plot_threshold_metrics(y_val, y_val_proba, save_path='reports/figures/threshold_tuning.png')
```

**Expected optimal threshold:** 0.10 – 0.25 (much lower than 0.5).

### 4.5 Calibration Assessment

Even with high AUC, predicted probabilities may be miscalibrated. This is critical because PDs are used directly in expected-loss calculations.

```python
from src.evaluate import CalibrationAssessor

calibrator = CalibrationAssessor(method='isotonic')
cal_report = calibrator.assess_calibration(y_val, y_val_proba)

print(f'Brier Score: {cal_report["brier_score"]:.4f}')
print(f'ECE: {cal_report["expected_calibration_error"]:.4f}')
print(f'Calibration needed: {cal_report["needs_calibration"]}')

if cal_report['needs_calibration']:
    y_test_proba_calibrated = calibrator.calibrate(
        best_model, X_train, y_train, X_test
    )
```

**Expected outcome:**
- Logistic Regression: naturally calibrated (low ECE < 0.01)
- XGBoost / LightGBM: may have ECE of 0.02–0.05 (mild miscalibration)
- After isotonic calibration: ECE < 0.01
- AUC does NOT change after calibration (it is rank-based)

---

## Section 5: Visualizations

All plots are generated via `src.evaluate.EvaluationVisualizer`.

```python
viz = EvaluationVisualizer(output_dir='reports/figures/')

# Individual plots
viz.plot_roc_curve(y_test, y_pred_proba, save_path='reports/figures/roc_curve.png')
viz.plot_pr_curve(y_test, y_pred_proba, save_path='reports/figures/pr_curve.png')
viz.plot_confusion_matrix(y_test, y_pred_binary, save_path='reports/figures/confusion_matrix.png')
viz.plot_ks_statistic(y_test, y_pred_proba, save_path='reports/figures/ks_statistic.png')
viz.plot_calibration_curve(y_test, y_pred_proba, save_path='reports/figures/calibration_curve.png')
viz.plot_score_distribution(y_test, y_pred_proba, save_path='reports/figures/score_distribution.png')
viz.plot_lift_chart(y_test, y_pred_proba, save_path='reports/figures/lift_chart.png')

# Or generate all at once
viz.plot_all(y_test, y_pred_proba)
```

**Figure summary (saved to `reports/figures/`):**

| File | Purpose |
|---|---|
| `roc_curve.png` | ROC curve with AUC annotated; model vs. random classifier |
| `pr_curve.png` | Precision-Recall curve with baseline prevalence line |
| `confusion_matrix.png` | TP/FP/TN/FN counts at the optimal threshold |
| `ks_statistic.png` | Cumulative distribution of scores for default vs. repay classes |
| `calibration_curve.png` | Reliability diagram with perfect-calibration reference |
| `threshold_tuning.png` | Precision/Recall/F1/F2 as functions of threshold |
| `score_distribution.png` | Histogram of predicted probabilities, split by true class |
| `lift_chart.png` | Cumulative default capture rate vs. random selection (by decile) |

---

## Section 6: Model Persistence

All training artifacts are saved by `ModelPersister`.

```python
from src.train import ModelPersister

persister = ModelPersister()
experiment_path = persister.save({
    'model': best_model,
    'scaler': scaler,
    'feature_names': list(X_train.columns),
    'config': config,
    'cv_scores': cv_scores,
    'leaderboard': leaderboard,
    'test_metrics': eval_results['metrics'],
    'optimal_threshold': optimal_threshold,
}, experiment_name='credit_risk_v1')

print(f'Model saved to: {experiment_path}')
```

**Saved structure:**

```
models/
└── 20260708_143000_credit_risk_v1/
    ├── model.pkl              # Trained classifier
    ├── scaler.pkl             # Fitted RobustScaler
    ├── feature_names.json     # Feature names
    ├── config.json            # Training configuration
    ├── cv_results.json        # Cross-validation scores
    ├── leaderboard.csv        # Model comparison table
    ├── test_metrics.json      # Final test set metrics
    ├── optimal_threshold.json # Chosen threshold
    └── optuna_study.pkl       # Hyperparameter optimization study
```

---

## Section 7: Model Card

The model card is a concise human-readable summary of the model's purpose, performance, limitations, and intended use. It is generated by `src.evaluate.ModelCardGenerator`.

```python
from src.evaluate import ModelCardGenerator

card = ModelCardGenerator(
    model_name='XGBoost_20260708',
    metrics=eval_results['metrics'],
    threshold=optimal_threshold,
    feature_importance=feature_importance,
    config=config,
)
card.generate_markdown(save_path='reports/model_card.md')
```

**Model card sections:**
1. Model Overview (type, training date, feature count)
2. Intended Use (portfolio risk assessment, collections prioritization)
3. Performance (test-set metrics table)
4. Threshold (selected threshold and business rationale)
5. Calibration (ECE, Brier score, calibration curve reference)
6. Feature Importance (top 10 features by gain)
7. Fairness (disparate impact ratio, if applicable to the data)
8. Limitations (data shift risk, thin-file population, edge cases)
9. Recommendations (how to use, monitoring cadence, retraining schedule)

---

## Expected Results Summary

### Performance Targets

| Metric | Target | Expected (Best Model) |
|---|---|---|
| ROC-AUC | ≥ 0.80 | 0.80 – 0.83 |
| Gini Coefficient | ≥ 0.60 | 0.60 – 0.66 |
| KS Statistic | ≥ 0.45 | 0.45 – 0.52 |
| Average Precision | ≥ 0.35 | 0.38 – 0.45 |
| Precision@5% | ≥ 0.40 | 0.40 – 0.50 |
| Recall@5% | ≥ 30% | 30 – 40% |
| Brier Score | < 0.08 | 0.06 – 0.08 |
| Log Loss | < 0.28 | 0.25 – 0.28 |
| Optimal Threshold | — | 0.10 – 0.25 |

### Model Ranking Prediction

```
1st: XGBoost  (AUC ~0.81, best overall)
2nd: LightGBM (AUC ~0.80, close second, faster)
3rd: RandomForest (AUC ~0.78, strong but slower)
4th: LogisticRegression (AUC ~0.74, interpretable baseline)
5th: DecisionTree (AUC ~0.65, weak — mostly to show why ensembles win)
```

### Next Steps

After selecting the best model:
1. **Model Explainability** (`notebooks/06_model_explainability.ipynb`): SHAP analysis, partial dependence plots, fairness audit.
2. **Business Simulation** (`notebooks/07_business_simulation.ipynb`): Translate model predictions into dollar-denominated profit, threshold optimization, scenario analysis.
3. **Dashboard** (`dashboard/`): Interactive portfolio risk explorer for stakeholders.

---

## Appendix: Anti-Patterns Checklist

Before proceeding, verify:

- [ ] No test set data used in training, tuning, threshold selection, or calibration.
- [ ] All scalers/encoders fit on training data only; test data is only transformed.
- [ ] Stratified split preserves the 8% default rate in all sets.
- [ ] Cross-validation uses `StratifiedKFold`, not `KFold`.
- [ ] Hyperparameter tuning uses cross-validated AUC, not a single split.
- [ ] `scale_pos_weight` (XGBoost) and `class_weight` (logreg/rf) are set to handle imbalance.
- [ ] Early stopping uses the validation set, not the test set.
- [ ] Threshold tuning uses the validation set; test set threshold is derived, not tuned.
- [ ] Calibration fitting uses separate folds, not the same data used for training.
- [ ] Feature names in test set match feature names in training set exactly.
- [ ] All random seeds are fixed for reproducibility.
- [ ] Training time is logged and considered in model selection (not just AUC).
- [ ] Model artifacts are versioned and include config for reproducibility.